![Week 2 Day 5](../../assets/banners/w2d5.png)

# Week 2 Day 5 - SOLUTION - k, and Why Spanning Questions Are Hard

Reference solution for the exercise at the end of `week2/day5.ipynb`. Have a go yourself first: struggling a little is where the learning happens! Then compare.

**The exercise:** re-run the evaluation with `--k 3` and `--k 10`; use `--row` on a spanning question (58 or 59) and explain its score.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import subprocess
import sys

from gradewise.config import REPO_ROOT


def run(*args: str) -> None:
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "COLUMNS": "100"}
    result = subprocess.run(
        [sys.executable, *args], cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", env=env
    )
    print(result.stdout, result.stderr, sep="")

## Part 1: k = 3 versus k = 10

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25", "--k", "3")

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25", "--k", "10")

**What changes, and why:**
- **Keywords** (coverage) and **Source@k** can only go **up** as k grows: the top 10 contains the top 3, so anything found at k = 3 is still found at k = 10. The biggest jumps are in `spanning` and `holistic`, whose keywords are spread over several chunks.
- **MRR** barely changes: it only looks at the *first* hit for each keyword, which is usually already in the top 3. Extra ranks only help keywords that were missing entirely, and they contribute small amounts (1/8, 1/9…).
- **nDCG** can move either way: more relevant chunks lower down add a little gain, but the *ideal* ranking also gets more relevant chunks to place at the top, raising the bar.

The trade-off: k = 10 sends roughly three times as much text to the LLM as k = 3. More context means more cost and, past a point, more noise for the model to ignore.

## Part 2: A spanning question in detail

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25", "--row", "58")

In [ ]:
run("week2/evaluation/eval.py", "--offline", "--pipeline", "bm25", "--row", "59")

**Why they score low, in one sentence each:**
- **Row 58** ("why denormalise, and what do indexes cost on writes?"): the answer lives in **two sections**. The index-cost half (3.7 Indexing) only arrives at rank 4, and the denormalise half (a misconception in 3.5 Normalisation) never makes the top 5, because the question's common words ("system", "tables") pull BM25 towards chunks that merely repeat them, such as the learning outcomes and 1.1 *What an operating system does*.
- **Row 59** ("how does TCP achieve reliable delivery over an unreliable network?"): all three keywords are found now, but not at rank 1: 2.7 *Network performance* says "reliable" and "unreliable" more often than the chunk that lists sequence numbers and ACKs. 🔁 This row is also how we found a bug: the tokenizer used to do **no stemming**, so "reliable" never matched "How TCP achieves *reliability*" and only 1 of the 3 keywords was found. `tokenize()` now stems chunks and queries alike, which fixed that half of the problem.

What's left calls for different tools: row 58 is a textbook case for **multi-query** (one rewrite per sub-question), and row 59 for **reranking** or **dense retrieval**, which judge what a chunk is *about* rather than how often it repeats a word. Hybrid search, and week 4's agentic retrieval, are built for exactly these.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Everything here is <code>week2/evaluation/eval.py</code>. Try <code>--compare</code> with a key to see simple → pro with real embeddings.</span>
        </td>
    </tr>
</table>